## transactions 테이블 생성

### 일반 거래 + completed와 연결된 거래

In [1]:
import pandas as pd
import numpy as np

# ==========================================================
# 1) 데이터 불러오기
# ==========================================================
# merged_df는 completed, transaction, viewed, received 등이 모두 들어있는
# 이벤트 로그 형태의 데이터라고 가정
df = pd.read_csv("../../Data/merged_df_260325.csv")


# ==========================================================
# 2) completed 행만 따로 추출
# ==========================================================
# transaction과 연결할 대상은 completed 이벤트이므로
# event가 completed인 행만 먼저 따로 뽑아둠
completed_df = df[df['event'] == 'completed'].copy()


# ==========================================================
# 3) completed 정보를 고객 + 시간 기준으로 요약
# ==========================================================
# 목적:
# 같은 고객(customer_id)이 같은 시간(time)에 completed를 남긴 경우를
# 하나의 묶음으로 보고 transaction과 연결하기 위해 요약 테이블을 만듦
#
# 왜 필요한가?
# 원본 데이터에서는 completed와 transaction이 서로 다른 행으로 기록되어 있음
# 그래서 transaction 행 기준으로 분석하려면
# "같은 고객이 같은 시간에 completed도 있었는지"를 붙여줘야 함
#
# 각 컬럼 설명:
# - completed_count:
#     같은 고객-같은 시간에 completed가 몇 건 있었는지
#     -> is_completed_linked 판단 기준으로 사용
#
# - offer_id / offer_label / reward / difficulty / duration / 채널 정보:
#     completed가 1개로 명확하면 그 값을 transaction에 붙임
#     completed가 여러 건이어서 값이 섞이면 np.nan 처리
#     -> 애매한 경우 억지로 잘못 붙이지 않기 위해서
#
# - bonus_reward:
#     보상값이 여러 개일 수 있으므로 일단 max 사용
#     보통 completed 보상은 같은 건에서 동일하거나 하나만 있는 경우가 많음
completed_summary = (
    completed_df
    .groupby(['customer_id', 'time'], as_index=False)
    .agg(
        # 같은 고객-같은 시간에 completed가 총 몇 건 있었는지
        completed_count=('event', 'size'),

        # offer_id가 하나로 명확할 때만 그 값을 사용
        offer_id=(
            'offer_id',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),

        # bonus_reward는 대표값으로 최댓값 사용
        bonus_reward=('bonus_reward', 'max'),

        # 아래 항목들도 값이 하나로 명확할 때만 채움
        reward=(
            'reward',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        difficulty=(
            'difficulty',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        duration=(
            'duration',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        web=(
            'web',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        email=(
            'email',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        mobile=(
            'mobile',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        social=(
            'social',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        ),
        offer_label=(
            'offer_label',
            lambda x: x.dropna().iloc[0] if x.dropna().nunique() == 1 else np.nan
        )
    )
)


# ==========================================================
# 4) transaction 행만 따로 추출
# ==========================================================
# 최종적으로 만들고 싶은 파일은 transaction 전용 csv이므로
# 원본에서 transaction 행만 따로 뽑음
transactions = df[df['event'] == 'transaction'].copy()


# ==========================================================
# 5) transaction 행에 completed 요약 정보 붙이기
# ==========================================================
# 연결 기준:
# 같은 customer_id + 같은 time
#
# 즉,
# 같은 고객이 같은 시점에 transaction도 했고 completed도 했다면
# 그 transaction은 completed와 연결된 거래로 간주
#
# how='left' 이므로
# transaction은 모두 남기고
# 연결되는 completed 정보가 있으면 옆에 붙음
transactions = transactions.merge(
    completed_summary,
    on=['customer_id', 'time'],
    how='left',
    suffixes=('', '_from_completed')
)


# ==========================================================
# 6) completed와 연결된 transaction인지 표시
# ==========================================================
# 여기서는 두 번째 기준을 반영:
# completed_count가 1 이상이면
# -> 같은 고객 + 같은 시간에 completed가 있었다는 뜻
# -> 연결된 transaction으로 표시
#
# 주의:
# 이 값은 "오퍼 정보가 명확하게 채워졌는지"와는 별개임
# completed가 여러 건이라 offer_id는 못 채워도
# completed 자체가 있었으면 is_completed_linked = 1
transactions['is_completed_linked'] = np.where(
    transactions['completed_count'].fillna(0) >= 1,
    1,
    0
)


# ==========================================================
# 7) event 값은 transaction으로 유지
# ==========================================================
# 이 파일은 completed 원본 로그가 아니라
# "transaction 기준 분석용 테이블"이므로 event는 그대로 transaction 유지
transactions['event'] = 'transaction'


# ==========================================================
# 8) transaction 행의 빈 오퍼 정보를 completed 정보로 채우기
# ==========================================================
# 목적:
# 원래 transaction 행에는 offer 관련 정보가 비어 있는 경우가 많음
# 그런데 같은 시점에 completed가 있었다면
# 그 completed의 오퍼 정보를 transaction 행에 옮겨 담아
# 분석하기 쉽게 만들 수 있음
#
# 원칙:
# - transaction 원래 값이 있으면 그대로 둠
# - transaction 값이 비어 있으면 completed에서 가져온 값으로 채움
#
# 예:
# transaction 행의 offer_id가 NaN이고,
# completed에서 offer_id가 명확하면 그 값으로 채움
fill_cols = [
    'offer_id',
    'bonus_reward',
    'reward',
    'difficulty',
    'duration',
    'web',
    'email',
    'mobile',
    'social',
    'offer_label'
]

for col in fill_cols:
    completed_col = col + '_from_completed'
    if completed_col in transactions.columns:
        transactions[col] = transactions[col].fillna(transactions[completed_col])


# ==========================================================
# 9) merge하면서 잠깐 사용한 보조 컬럼 제거
# ==========================================================
# _from_completed로 끝나는 컬럼들은
# completed에서 붙여온 임시 컬럼들이므로 정리해서 제거
#
# completed_count는 is_completed_linked 계산에는 썼지만
# 최종 csv에서 꼭 필요하지 않다면 제거
drop_cols = [col for col in transactions.columns if col.endswith('_from_completed')]
drop_cols += ['completed_count']

transactions = transactions.drop(columns=drop_cols, errors='ignore')


# ==========================================================
# 10) 최종 컬럼 순서 정리
# ==========================================================
# merged_df와 최대한 같은 구조를 유지하면서
# 맨 뒤에 is_completed_linked만 추가
final_cols = [
    'person',
    'event',
    'time',
    'offer_id',
    'amount',
    'bonus_reward',
    'time_days',
    'gender',
    'age',
    'customer_id',
    'became_member_on',
    'income',
    'income_missing',
    'age_missing',
    'reward',
    'difficulty',
    'duration',
    'web',
    'email',
    'mobile',
    'social',
    'offer_label',
    'is_completed_linked'
]

transactions = transactions[final_cols]


# ==========================================================
# 11) 결과 확인
# ==========================================================
print("=" * 60)
print("최종 transactions 행 수:", len(transactions))
print("=" * 60)

print("completed 연결 여부 분포:")
print(transactions['is_completed_linked'].value_counts(dropna=False))
print("=" * 60)

print(transactions.head())


# ==========================================================
# 12) csv 저장
# ==========================================================
transactions.to_csv("../../Data/transactions.csv", index=False)
print("저장 완료: ../../Data/transactions.csv")

최종 transactions 행 수: 138953
completed 연결 여부 분포:
is_completed_linked
0    108336
1     30617
Name: count, dtype: int64
                             person        event  time  \
0  0009655768c64bdeb2e877511632db8f  transaction   228   
1  0009655768c64bdeb2e877511632db8f  transaction   414   
2  0009655768c64bdeb2e877511632db8f  transaction   528   
3  0009655768c64bdeb2e877511632db8f  transaction   552   
4  0009655768c64bdeb2e877511632db8f  transaction   576   

                           offer_id  amount  bonus_reward  time_days gender  \
0                               NaN   22.16           NaN         10      M   
1  f19421c1d4aa40978ebb69ca19b0e20d    8.57           5.0         18      M   
2  fafdcd668e3743c1bb461111dcafc2a4   14.11           2.0         23      M   
3                               NaN   13.56           NaN         24      M   
4  2906b810c7d4411798c6938adc9daaa5   10.27           2.0         25      M   

    age                       customer_id  ... age_missing

In [2]:
transactions.shape

(138953, 23)

In [3]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 138953 entries, 0 to 138952
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   person               138953 non-null  str    
 1   event                138953 non-null  str    
 2   time                 138953 non-null  int64  
 3   offer_id             28112 non-null   str    
 4   amount               138953 non-null  float64
 5   bonus_reward         30617 non-null   float64
 6   time_days            138953 non-null  int64  
 7   gender               138953 non-null  str    
 8   age                  123957 non-null  float64
 9   customer_id          138953 non-null  str    
 10  became_member_on     138953 non-null  str    
 11  income               123957 non-null  float64
 12  income_missing       138953 non-null  int64  
 13  age_missing          138953 non-null  int64  
 14  reward               28599 non-null   float64
 15  difficulty           28696 n

In [4]:
transactions.head(20)

,person,event,time,offer_id,amount,bonus_reward,time_days,gender,age,customer_id,...,age_missing,reward,difficulty,duration,web,email,mobile,social,offer_label,is_completed_linked
0,0009655768c64bdeb2e877511632db8f,transaction,228,NaN,22.16,NaN,10,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,0009655768c64bdeb2e877511632db8f,transaction,414,f19421c1d4aa40978ebb69ca19b0e20d,8.57,5.0,18,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,5.0,5.0,5.0,1.0,1.0,1.0,1.0,bogo_4,1
2,0009655768c64bdeb2e877511632db8f,transaction,528,fafdcd668e3743c1bb461111dcafc2a4,14.11,2.0,23,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,2.0,10.0,10.0,1.0,1.0,1.0,1.0,discount_3,1
3,0009655768c64bdeb2e877511632db8f,transaction,552,NaN,13.56,NaN,24,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,0009655768c64bdeb2e877511632db8f,transaction,576,2906b810c7d4411798c6938adc9daaa5,10.27,2.0,25,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,2.0,10.0,7.0,1.0,1.0,1.0,0.0,discount_4,1
5,0009655768c64bdeb2e877511632db8f,transaction,660,NaN,12.36,NaN,28,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6,0009655768c64bdeb2e877511632db8f,transaction,690,NaN,28.16,NaN,29,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
7,0009655768c64bdeb2e877511632db8f,transaction,696,NaN,18.41,NaN,30,M,33.0,0009655768c64bdeb2e877511632db8f,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
8,00116118485d4dfda04fdbaba9a87b5c,transaction,294,NaN,0.70,NaN,13,Unknown,NaN,00116118485d4dfda04fdbaba9a87b5c,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
9,00116118485d4dfda04fdbaba9a87b5c,transaction,456,NaN,0.20,NaN,20,Unknown,NaN,00116118485d4dfda04fdbaba9a87b5c,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


전체 transactions 매출: $1,775,451  
전체 completed 매출: $680,307

In [5]:
transactions['amount'].sum()

np.float64(1775451.97)

In [6]:
transactions[transactions['is_completed_linked'] == 1]['amount'].sum()

np.float64(616623.6799999999)